# Adversarial TTS - Class-Based Architecture

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

## 1. Imports

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

In [ ]:
%cd StyleTTS2

import torch
import argparse
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt_tab')

from Datastructures.dataclass import ModelData
from Objectives.FitnessObjective import FitnessObjective

# Import class-based modules
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger
from Trainer.VectorManipulator import VectorManipulator

# Import Pymoo components
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
%%bash
git pull

## 2. Configuration

Edit the values below to configure the optimization run.

In [ ]:
# =============================================================================
# Configuration - Edit these values directly
# =============================================================================
args = argparse.Namespace(
    # Text parameters
    ground_truth_text="The birch canoe slid on the smooth planks.",
    target_text="",

    # Optimization parameters
    loop_count=1,
    num_generations=100,
    pop_size=200,
    iv_scalar=0.5,
    size_per_phoneme=1,
    batch_size=100,  # -1 for full batch

    # Flags
    notify=False,
    subspace_optimization=False,

    # Mode and objectives
    mode="NOISE_UNTARGETED",  # TARGETED, UNTARGETED, NOISE_UNTARGETED
    objectives="PESQ=0.4, SET_OVERLAP=0.0",
)

# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(device)
config_data = loader.load_configuration(args)
config_data.print_summary()

tts_model, asr_model = loader.load_required_models()

audio_gt, audio_target, audio_embedding_gt, audio_embedding_target = loader.generate_audio_data(
    config_data.mode, config_data.text_gt, config_data.text_target, tts_model
)

print("\nEnvironment initialized successfully!")

# Initialize Objectives
objectives = loader.initialize_objectives(
    active_objectives=config_data.active_objectives,
    model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
    text_gt=config_data.text_gt,
    text_target=config_data.text_target,
    mode=config_data.mode,
    audio_gt=audio_gt,
)

vector_manipulator = VectorManipulator(audio_embedding_gt, audio_embedding_target.h_text, config_data)

# Create Trainer and Logger
trainer = AdversarialTrainer(
    tts_model, asr_model, config_data.thresholds, objectives, vector_manipulator, device
)
logger = RunLogger(
    config_data.active_objectives, tts_model, asr_model, vector_manipulator, device
)

print("Trainer components initialized!")

## 5. Run Optimization

Run the optimization loop and log results.

In [ ]:
for loop_iteration in range(config_data.loop_count):
    print(f"\n[Loop {loop_iteration + 1}/{config_data.loop_count}] Starting optimization loop...")

    # Initialize fresh optimizer for this cycle
    optimizer = PymooOptimizer(
        bounds=(0, 1),
        algorithm=NSGA2,
        algo_params={"pop_size": config_data.pop_size},
        num_objectives=len(config_data.active_objectives),
        solution_shape=(audio_embedding_gt.input_length.detach().cpu().item(), config_data.size_per_phoneme),
    )

    fitness_data, archive_data, generation_count, elapsed_time_total = trainer.run_full_iteration(optimizer, config_data.num_generations, config_data.pop_size, config_data.batch_size)

    # 5. Save all results (audios, spectrograms, graphs, etc.)
    logger.save_all_results(
        optimizer, fitness_data, archive_data, generation_count, elapsed_time_total,
        audio_gt, audio_target, config_data
    )

    print("[Log] Finished saving all results")

## 6. Download Results (Optional)

Set `download_results = True` to create a zip of the outputs folder and download it.

**On Colab:** Downloads via browser.  
**On Kaggle:** Creates zip in `/kaggle/working/` (available in Output tab after run completes).

In [ ]:
if True:
    import shutil
    import os
    
    # Detect environment
    on_kaggle = os.path.exists('/kaggle/working')
    
    if on_kaggle:
        # Kaggle: zip outputs to /kaggle/working (will appear in Output tab)
        shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/StyleTTS2/outputs')
        print("Created /kaggle/working/outputs.zip - available in Output tab after run completes")
    else:
        # Colab: zip and download via browser
        # from google.colab import files
        shutil.make_archive('/content/outputs', 'zip', '/content/StyleTTS2/outputs')
        # files.download('/content/outputs.zip')